# ManoVarta Aya Eval

This notebook downloads the saved Aya adapter bundle, runs resumable held-out evaluation, saves progress after every example, and stops immediately on the first parse failure so we can fix the parser before spending more compute.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_dir = Path('/content/ManoVarta')
if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/RitwijParmar/ManoVarta.git', str(repo_dir)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{repo_dir}[train]'], check=True)
print(subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD']).decode().strip())


In [ ]:
import os
from pathlib import Path

persist_root = Path(os.environ.get('MANOVARTA_PERSIST_ROOT', '/content/ManoVartaAyaEval'))
if os.environ.get('MANOVARTA_USE_DRIVE', '1') == '1':
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        persist_root = Path('/content/drive/MyDrive/ManoVartaAyaEval/current')
    except Exception as exc:
        print('Drive unavailable, using local storage:', exc)
persist_root.mkdir(parents=True, exist_ok=True)
bundle_root = persist_root / 'bundle'
bundle_root.mkdir(parents=True, exist_ok=True)
print(persist_root)


## Set your Hugging Face token
The token needs read access to `CohereLabs/aya-expanse-8b` and `ritwijar/manovarta-aya-eval-artifacts`.


In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN', '')
assert HF_TOKEN, 'Set HF_TOKEN first.'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = str(persist_root / '.hf_home')
print('HF token configured')


In [ ]:
import tarfile
from huggingface_hub import hf_hub_download

downloaded = hf_hub_download(
    repo_id='ritwijar/manovarta-aya-eval-artifacts',
    filename='aya_eval_upload.tar.gz',
    token=HF_TOKEN,
    local_dir=str(bundle_root),
    local_dir_use_symlinks=False,
)
print('downloaded', downloaded)
with tarfile.open(downloaded, 'r:gz') as tf:
    tf.extractall(bundle_root)
inner = bundle_root / 'manovarta_eval_min.tar.gz'
if inner.exists():
    with tarfile.open(inner, 'r:gz') as tf:
        tf.extractall(bundle_root / 'repo_payload')

adapter_dir = bundle_root / 'aya_bundle'
assert adapter_dir.exists(), adapter_dir
print('adapter', adapter_dir)


In [ ]:
import json

repo_dir = Path('/content/ManoVarta')
out_dir = persist_root / 'reports' / 'aya_eval'
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(repo_dir / 'tools' / 'resumable_extractor_eval.py'),
    '--model-path', str(adapter_dir),
    '--eval-file', str(repo_dir / 'data' / 'processed' / 'extractor_test.jsonl'),
    '--output-dir', str(out_dir),
    '--device', 'cuda',
    '--max-new-tokens', '1200',
    '--stop-on-parse-failure',
]
print('running', ' '.join(cmd))
result = subprocess.run(cmd, cwd=repo_dir, text=True)
print('returncode', result.returncode)
summary_path = out_dir / 'summary.json'
if summary_path.exists():
    print(summary_path.read_text())
if result.returncode not in (0, 2):
    raise RuntimeError(f'eval failed with code {result.returncode}')


In [ ]:
summary_path = persist_root / 'reports' / 'aya_eval' / 'summary.json'
progress_path = persist_root / 'reports' / 'aya_eval' / 'progress.jsonl'
print('summary exists', summary_path.exists())
print('progress exists', progress_path.exists())
if progress_path.exists():
    lines = progress_path.read_text(encoding='utf-8').splitlines()
    print('completed examples', len(lines))
    if lines:
        print(lines[-1])
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))


## Resume after a parser fix
Rerun the evaluation cell above after updating the repo. Completed examples in `progress.jsonl` will be skipped automatically.
